# Semana 4: clasificaci?n tabular de libros

Adaptamos el flujo de ?rboles de decisi?n al conjunto Goodreads. La etiqueta son categor?as derivadas de la calificaci?n (no etiquetas originales). Las caracter?sticas son metadatos disponibles del libro; no se usa `average_rating` como predictor. Comparamos un baseline mayoritario con un ?rbol ajustado por validaci?n cruzada, evaluando el conjunto de prueba una sola vez.

## 1. Carga y control de calidad

La auditor?a se ejecuta sobre los datos crudos. Los valores extremos de p?ginas se reportan y se excluyen solo para este modelo; no se borran del dataset exploratorio. Los faltantes de p?ginas se conservan para imputarlos dentro del pipeline.

In [ ]:
import sys
from datetime import datetime
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src" / "goodreads.py").exists():
    ROOT = ROOT.parent
if not (ROOT / "src" / "goodreads.py").exists():
    raise FileNotFoundError("No se encontr? src/goodreads.py. Abre el notebook desde el proyecto.")
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier

from goodreads import goodreads_quality_audit, prepare_goodreads_frame
from hf_loader import HFDatasetLoader

SEMILLA = 42

loader = HFDatasetLoader(dataset_name="pfaha/goodreads-books", split="train")
df_original = loader.load()
display(goodreads_quality_audit(df_original))

books = prepare_goodreads_frame(
    df_original,
    minimum_rating=3.5,
    maximum_rating=5.0,
    deduplicate_text=True,
)
print(f"Filas v?lidas sin t?tulos+autores duplicados: {len(books):,}")

## 2. Preparar objetivo y caracter?sticas

Las tres clases siguen los cortes usados en la semana 3. Se limita `paginas` al intervalo positivo hasta su percentil 99 para que unos pocos valores absurdos no dicten el entrenamiento. El pipeline imputar? faltantes usando solo los datos de entrenamiento.

In [ ]:
CLASES = ["calificacion_baja", "calificacion_media", "calificacion_alta"]
books["clase_calificacion"] = pd.cut(
    books["calificacion"],
    bins=[-np.inf, 4.0, 4.25, np.inf],
    labels=CLASES,
    right=False,
)

paginas_positivas = books.loc[books["paginas"].gt(0), "paginas"]
limite_paginas = float(paginas_positivas.quantile(0.99))
model_data = books[
    books["clase_calificacion"].notna()
    & (books["paginas"].isna() | books["paginas"].between(1, limite_paginas))
].copy()
model_data["anio_publicacion"] = pd.to_datetime(
    model_data["fecha_publicacion"], format="mixed", errors="coerce"
).dt.year
model_data["es_serie"] = (
    model_data["es_serie"].astype("string").str.lower().map({"true": 1, "false": 0, "1": 1, "0": 0})
)

FEATURES = ["paginas", "anio_publicacion", "cantidad_ediciones", "es_serie"]
missing_features = set(FEATURES).difference(model_data.columns)
if missing_features:
    raise KeyError(f"Faltan caracter?sticas en Goodreads: {sorted(missing_features)}")
model_data = model_data.dropna(subset=["clase_calificacion"])
model_data = model_data.sample(n=min(30_000, len(model_data)), random_state=SEMILLA)
X = model_data[FEATURES].apply(pd.to_numeric, errors="coerce")
y = model_data["clase_calificacion"].astype(str)
print(f"L?mite p?ginas P99: {limite_paginas:.0f}")
print("Distribuci?n de clases:\n", y.value_counts().reindex(CLASES))

## 3. Separaci?n, baseline y selecci?n del ?rbol

La partici?n es estratificada y reproducible. La b?squeda de profundidad/hoja usa solo entrenamiento; la prueba queda reservada para la evaluaci?n final.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEMILLA, stratify=y
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)
pipeline = Pipeline([
    ("imputacion", SimpleImputer(strategy="median", add_indicator=True)),
    ("arbol", DecisionTreeClassifier(class_weight="balanced", random_state=SEMILLA)),
])
busqueda = GridSearchCV(
    pipeline,
    {
        "arbol__max_depth": [2, 3, 5, 8],
        "arbol__min_samples_leaf": [20, 50],
    },
    scoring="f1_macro",
    cv=cv,
    n_jobs=1,
    refit=True,
)
busqueda.fit(X_train, y_train)

baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
pred_baseline = baseline.predict(X_test)
pred = busqueda.predict(X_test)
metricas = {
    "exactitud_baseline": accuracy_score(y_test, pred_baseline),
    "f1_macro_baseline": f1_score(y_test, pred_baseline, average="macro"),
    "exactitud_arbol": accuracy_score(y_test, pred),
    "f1_macro_arbol": f1_score(y_test, pred, average="macro"),
    "f1_macro_cv": busqueda.best_score_,
}
print("Mejores par?metros:", busqueda.best_params_)
print(pd.Series(metricas).round(4))
print(classification_report(y_test, pred, labels=CLASES, zero_division=0, digits=4))
matriz = confusion_matrix(y_test, pred, labels=CLASES)
display(pd.DataFrame(matriz, index=CLASES, columns=CLASES))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
image = ax.imshow(matriz, cmap="Blues")
ax.set(xticks=range(len(CLASES)), xticklabels=CLASES,
       yticks=range(len(CLASES)), yticklabels=CLASES,
       xlabel="Predicci?n", ylabel="Real", title="Matriz de confusi?n")
plt.setp(ax.get_xticklabels(), rotation=25, ha="right")
for i in range(len(CLASES)):
    for j in range(len(CLASES)):
        ax.text(j, i, f"{matriz[i, j]:,}", ha="center", va="center")
fig.colorbar(image, ax=ax)
plt.tight_layout()
plt.show()

## 4. Guardar resultados

El reporte guarda los resultados de esta partici?n y la configuraci?n seleccionada. El modelo se conserva en `modelos/libros/`, ruta excluida de Git. Una m?trica mejor que el baseline no implica causalidad ni que Goodreads conozca la calificaci?n antes de publicarse.

In [ ]:
from joblib import dump

report_path = ROOT / "reportes" / "resumen_clasificacion_libros.txt"
report_path.parent.mkdir(parents=True, exist_ok=True)
model_path = ROOT / "modelos" / "libros" / f"{datetime.now():%Y%m%d_%H%M%S}_arbol_goodreads.pkl"
model_path.parent.mkdir(parents=True, exist_ok=True)
dump(busqueda.best_estimator_, model_path)
report_path.write_text(
    "CLASIFICACI?N TABULAR GOODREADS\n"
    "================================\n"
    f"Filas originales: {len(df_original):,}\n"
    f"Filas para modelado: {len(model_data):,}; semilla: {SEMILLA}; prueba: 20%\n"
    f"Caracter?sticas: {FEATURES}\n"
    f"L?mite de p?ginas P99: {limite_paginas:.2f}\n"
    f"Distribuci?n de clases:\n{y.value_counts().reindex(CLASES).to_string()}\n\n"
    f"Par?metros elegidos por validaci?n cruzada: {busqueda.best_params_}\n"
    f"M?tricas: {pd.Series(metricas).round(4).to_string()}\n\n"
    f"Reporte por clase:\n{classification_report(y_test, pred, labels=CLASES, zero_division=0, digits=4)}\n"
    f"Matriz (filas=real, columnas=predicci?n):\n{pd.DataFrame(matriz, index=CLASES, columns=CLASES).to_string()}\n"
    "Nota: las clases son intervalos derivados de average_rating, no etiquetas originales.\n",
    encoding="utf-8",
)
print(f"Reporte: {report_path}")
print(f"Modelo guardado (ignorado por Git): {model_path}")